# `explainability` -- SHAP Feature Importance for Calibrated Models

**Ported from `project_gagan` notebook cell 24.** The original notebook's SHAP cell explained a larger "extended features" model zoo (with venue and batting-quality features) that wasn't carried into this merge's default pipeline -- so this module preserves the *technique*, generalised into a reusable function that works with any calibrated classifier and feature set, applied in `run_all.py`/`notebooks/run_all.ipynb` to the pre-match Cal. GBT model.

**The one thing this module is careful to get right (DEF-H02 fix, preserved from the original):** it explains the **calibrated** model's `predict_proba` output, not the raw uncalibrated base estimator underneath it. `CalibratedClassifierCV` wraps a tree/logistic model with an isotonic-regression recalibration layer -- if you only explained the raw tree, the calibration layer's effect on the final probability would be completely invisible to the SHAP analysis, which is misleading for exactly the models this project cares most about (every classifier here is calibrated).

In [1]:
import numpy as np

SEED = 42

## `shap_importance()`

**Why permutation, not TreeExplainer/LinearExplainer**: those faster, model-specific SHAP explainers only work on the raw base estimator, which is exactly the DEF-H02 problem described above. `shap.Explainer(..., algorithm="permutation")` treats the model as a black box -- it only needs a callable that maps feature rows to predictions -- so it works correctly no matter how many calibration/wrapper layers sit on top of the base estimator. The tradeoff is speed: permutation SHAP is slower than the model-specific explainers, which is why this function subsamples both the background distribution (`n_background`, default 80) and the set of rows actually explained (`n_explain`, default 80) rather than using the full dataset.

**Steps:**
1. `shap.sample(...)` draws a random background sample from    `X_background` (typically the training set) -- this    represents "what a typical input looks like", which    permutation SHAP needs as a reference point for computing    each feature's marginal contribution.
2. `_prob(X)` is a thin wrapper exposing only the positive-class    probability (`predict_proba(X)[:, 1]`) as a single-output    function, since SHAP explains scalar outputs.
3. Run the explainer on (a subsample of) `X_explain` (typically    the test set) to get one SHAP value per feature per row.
4. Aggregate to **mean absolute SHAP value per feature** -- a    single importance ranking, sorted descending, which is what    `run_all.py` prints directly.

Both the raw per-row SHAP values and the aggregated importance ranking are returned, so a caller could build a beeswarm plot from `shap_values` if desired, even though this project's default entry points only print the aggregated ranking.

In [2]:
def shap_importance(model, X_background: np.ndarray, X_explain: np.ndarray,
                     feature_names: list[str], n_background: int = 80,
                     n_explain: int = 80) -> dict:
    """
    Compute mean |SHAP value| per feature for a calibrated classifier's
    positive-class probability.

    Parameters
    ----------
    model : a fitted classifier with predict_proba (e.g. CalibratedClassifierCV)
    X_background : array used to build the permutation background distribution
    X_explain : array of samples to explain
    feature_names : names matching the columns of X_background/X_explain

    Returns
    -------
    dict with keys 'shap_values' (array, n_explain x n_features) and
    'importance' (dict feature_name -> mean |SHAP value|, sorted descending).
    """
    import shap

    bg = shap.sample(X_background, min(n_background, len(X_background)), random_state=SEED)

    def _prob(X):
        return model.predict_proba(X)[:, 1]

    explainer = shap.Explainer(_prob, bg, algorithm="permutation")
    n = min(n_explain, len(X_explain))
    values = explainer(X_explain[:n]).values

    mean_abs = np.abs(values).mean(axis=0)
    importance = dict(
        sorted(zip(feature_names, mean_abs), key=lambda kv: -kv[1])
    )
    return {"shap_values": values, "importance": importance}